In [1]:
import sqlite3
from app.config import settings

conn = sqlite3.connect(settings.ytb_subtitles_db_url)
c = conn.cursor()

# Create tables
c.execute("""
CREATE TABLE IF NOT EXISTS videos (
    id TEXT PRIMARY KEY,
    url TEXT
)
""")
c.execute("""
CREATE VIRTUAL TABLE IF NOT EXISTS subtitles USING fts5(
    video_id,
    text,
    start,
    duration
)
""")
conn.commit()

In [2]:
# Initialize empty list
video_ids = []

# Open the file and read lines
with open("video_ids.txt", "r") as f:
    # Strip newline characters and add to list
    video_ids = [line.strip() for line in f if line.strip()]

print(len(video_ids))


1582


In [4]:
import sqlite3
import time
from app.services import ytb_preprocess, context_search
from app.config import settings
from youtube_transcript_api._errors import IpBlocked, NoTranscriptFound, TranscriptsDisabled
import random

conn = sqlite3.connect(settings.ytb_subtitles_db_url)

video_num = len(video_ids)
start_idx = 1576  # <-- start from this index
for idx, video_id in enumerate(video_ids[start_idx:], start=start_idx):
    try:
        context_search.add_subtitles_to_db(video_id=video_id, db=conn)
        start_idx += 1
        print(f"Added: {start_idx}/{video_num} -> {video_id}")
    except IpBlocked:
        print(f"IP blocked for video {video_id}, skipping for now.")
    except (NoTranscriptFound, TranscriptsDisabled) as e:
        print(f"No transcript available for {video_id}: {e}")
    except Exception as e:
        print(f"Unexpected error for {video_id}: {e}")

    # Add a delay between requests to be less aggressive
    time.sleep(random.uniform(30, 60))

# 88

Added: 1577/1582 -> 9oUJ0D9wlrw
Added: 1578/1582 -> 0yVViduOjog
Added: 1579/1582 -> vhlS5-P1cPk
Added: 1580/1582 -> sj27JBMRzZM
Added: 1581/1582 -> kOG2mBKC6Bw
Added: 1582/1582 -> 4Rv-QiAJnM0
